# Building Permits Leads

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/building-permits-leads.ipynb)

**What:** Building permits leads from official city open-data portals (Chicago, NYC, LA, SF, Seattle, Austin, Baton Rouge, Montgomery Co. MD, Cincinnati). Type, status, dates, valuation, property address, contractor business if published, source link; code violations too. Delta mode. $0.0015 per record.

**Why:** run the [`building-permits-leads`](https://apify.com/transparent_meteorite/building-permits-leads) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Contractor/applicant columns are not displayed; property-level fields only.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/building-permits-leads) for every option.

In [2]:
run_input = {   'cities': ['CHI', 'SEA', 'AUS'],
    'recordTypes': ['permit'],
    'sinceDays': 7,
    'minValuation': 0,
    'maxItems': 5}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/building-permits-leads').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.building-permits-leads runId:iY0qZoKjgE5yojYwB] -> Status: RUNNING, Message: 


[apify.building-permits-leads runId:iY0qZoKjgE5yojYwB] -> 2026-10-13T22:17:00.818Z ACTOR: Pulling container image of build RnTxDrI79bQ5Tv1ki from registry.


[apify.building-permits-leads runId:iY0qZoKjgE5yojYwB] -> 2026-10-13T22:17:01.751Z ACTOR: Creating container.


[apify.building-permits-leads runId:iY0qZoKjgE5yojYwB] -> 2026-10-13T22:17:01.839Z ACTOR: Starting container.


[apify.building-permits-leads runId:iY0qZoKjgE5yojYwB] -> 2026-10-13T22:17:01.840Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.building-permits-leads runId:iY0qZoKjgE5yojYwB] -> 2026-10-13T22:17:05.424Z Done. Delivered 5 records (charged 5). Warnings: 0.


[apify.building-permits-leads runId:iY0qZoKjgE5yojYwB] -> 2026-10-13T22:17:05.572Z INFO  [Status message]: Done: 5 items


[apify.building-permits-leads runId:iY0qZoKjgE5yojYwB] -> 2026-10-13T22:17:05.593Z Thanks for using Building Permits Leads. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/XhTgNUTP7fFiKhBBN/issues) with the run URL and we will reply there.


[apify.building-permits-leads runId:iY0qZoKjgE5yojYwB] -> Status: SUCCEEDED, Message: Done: 5 items


Run status: SUCCEEDED | items: 5


## 4. Tabulate

Contractor/applicant columns are not displayed; property-level fields only.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["city", "state", "permitType", "status", "issueDate", "valuation", "propertyAddress"] if c in df.columns]
df[SHOW]

,city,state,permitType,status,issueDate,valuation,propertyAddress
0,Chicago,IL,EXPRESS PERMIT PROGRAM,ACTIVE,2026-10-12,892000.0,"432 W 14TH ST, Chicago, IL"
1,Chicago,IL,EXPRESS PERMIT PROGRAM,ACTIVE,2026-10-12,NaN,"1301 W MADISON ST, Chicago, IL"
2,Seattle,WA,Demolition,Issued,2026-10-12,800000.0,"3505 DENSMORE AVE N, SEATTLE, WA 98103"
3,Seattle,WA,Building - New,Issued,2026-10-12,794238.0,"3505 DENSMORE AVE N, SEATTLE, WA 98103"
4,Austin,TX,Building Permit,Active,2026-10-12,1404500.0,"900 E PARMER LN BLDG 6, AUSTIN, TX 78753"


## 5. A quick analysis

In [5]:
df['valuation']=pd.to_numeric(df['valuation'],errors='coerce')
df.groupby('city')['valuation'].agg(['count','sum','max'])

,count,sum,max
city,,,
Austin,1,1404500.0,1404500.0
Chicago,1,892000.0,892000.0
Seattle,2,1594238.0,800000.0


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [building-permits-leads on the Apify Store](https://apify.com/transparent_meteorite/building-permits-leads).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.